<a href="https://colab.research.google.com/github/ssuazam/Arquisoft/blob/main/7.%20Gen%20Ai/Punto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Agente IA para Manejo de Tareas del Curso

> **100% Gratuito** · Funciona en Google Colab · Usa Google Gemini API (free tier)

## ¿Qué puede hacer este agente?
- ✅ **Crear y listar tareas** del curso
- 📅 **Gestionar fechas de entrega** y prioridades
- 🤖 **Responder preguntas** sobre tus tareas con IA
- 📊 **Generar resúmenes** del progreso del curso
- 💡 **Dar sugerencias** para organizar mejor tu tiempo
- 🔍 **Buscar tareas** por tema, estado o fecha

---
### Configuración inicial
1. Obtén tu API key **gratuita** en: https://aistudio.google.com/app/apikey
2. Ejecuta las celdas en orden
3. ¡Listo! El chat aparecerá al final


In [1]:
# ============================================================
# CELDA 1: Instalación de dependencias
# ============================================================
!pip install google-generativeai ipywidgets -q
print("✅ Dependencias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 127.3 kB/s eta 0:00:00
✅ Dependencias instaladas correctamente


In [2]:
# ============================================================
# CELDA 2: Configurar API Key de Google Gemini (GRATIS)
# ============================================================
# Obtén tu key gratuita en: https://aistudio.google.com/app/apikey

import google.generativeai as genai
from google.colab import userdata

# Opción A: Usar Secrets de Colab (recomendado)
# Ve a la llave 🔑 en el panel izquierdo > "Add new secret" > Nombre: GEMINI_API_KEY
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ API Key cargada desde Secrets de Colab")
except:
    # Opción B: Ingresar manualmente
    API_KEY = input("🔑 Ingresa tu API Key de Google Gemini: ").strip()

genai.configure(api_key=API_KEY)
print("✅ Gemini configurado correctamente")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ API Key cargada desde Secrets de Colab
✅ Gemini configurado correctamente


In [3]:
# ============================================================
# CELDA 3: Base de datos de tareas (en memoria con JSON)
# ============================================================
import json
import datetime
import os
from pathlib import Path

TAREAS_FILE = "tareas_curso.json"

def cargar_tareas():
    if Path(TAREAS_FILE).exists():
        with open(TAREAS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def guardar_tareas(tareas):
    with open(TAREAS_FILE, 'w', encoding='utf-8') as f:
        json.dump(tareas, f, ensure_ascii=False, indent=2)

def agregar_tarea(titulo, descripcion, fecha_entrega, materia, prioridad="media"):
    tareas = cargar_tareas()
    nueva = {
        "id": len(tareas) + 1,
        "titulo": titulo,
        "descripcion": descripcion,
        "fecha_entrega": fecha_entrega,
        "materia": materia,
        "prioridad": prioridad,
        "estado": "pendiente",
        "creada": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    tareas.append(nueva)
    guardar_tareas(tareas)
    return nueva

def completar_tarea(tarea_id):
    tareas = cargar_tareas()
    for t in tareas:
        if t["id"] == tarea_id:
            t["estado"] = "completada"
            t["completada_en"] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
            guardar_tareas(tareas)
            return t
    return None

def eliminar_tarea(tarea_id):
    tareas = cargar_tareas()
    tareas = [t for t in tareas if t["id"] != tarea_id]
    guardar_tareas(tareas)

def obtener_resumen():
    tareas = cargar_tareas()
    total = len(tareas)
    completadas = sum(1 for t in tareas if t["estado"] == "completada")
    pendientes = total - completadas
    hoy = datetime.date.today()
    proximas = []
    vencidas = []
    for t in tareas:
        if t["estado"] == "pendiente" and t.get("fecha_entrega"):
            try:
                fecha = datetime.datetime.strptime(t["fecha_entrega"], "%Y-%m-%d").date()
                dias = (fecha - hoy).days
                if dias < 0:
                    vencidas.append(t)
                elif dias <= 7:
                    proximas.append(t)
            except:
                pass
    return {
        "total": total, "completadas": completadas,
        "pendientes": pendientes, "proximas_7dias": proximas, "vencidas": vencidas
    }

# Cargar datos de ejemplo si no hay tareas
if not cargar_tareas():
    tareas_ejemplo = [
        ("Parcial de Cálculo", "Estudiar derivadas e integrales para el examen", "2025-04-15", "Cálculo", "alta"),
        ("Ensayo de Historia", "Escribir 3 páginas sobre la Revolución Francesa", "2025-04-10", "Historia", "alta"),
        ("Laboratorio de Física", "Informe del experimento de péndulo", "2025-04-20", "Física", "media"),
        ("Lectura de Filosofía", "Leer capítulos 4 y 5 de Kant", "2025-04-18", "Filosofía", "baja"),
        ("Proyecto de Programación", "Implementar algoritmo de ordenamiento", "2025-04-25", "Programación", "alta"),
    ]
    for t in tareas_ejemplo:
        agregar_tarea(*t)
    print("📚 Tareas de ejemplo cargadas")

print("✅ Sistema de tareas listo. Total de tareas:", len(cargar_tareas()))

📚 Tareas de ejemplo cargadas
✅ Sistema de tareas listo. Total de tareas: 5


In [4]:
# ============================================================
# CELDA 4: Definición del Agente IA
# ============================================================

SYSTEM_PROMPT = """
Eres un agente IA especializado en ayudar estudiantes a manejar sus tareas universitarias.
Tu nombre es ProfeBot 🎓.

Tienes acceso al sistema de tareas del estudiante. Cuando el usuario te pida:

1. VER TAREAS: Lista las tareas de forma organizada por prioridad o fecha
2. AGREGAR TAREA: Extrae titulo, descripcion, fecha_entrega (YYYY-MM-DD), materia y prioridad (alta/media/baja)
   Responde con JSON: {"accion": "agregar", "titulo": "...", "descripcion": "...", "fecha_entrega": "YYYY-MM-DD", "materia": "...", "prioridad": "..."}
3. COMPLETAR TAREA: Identifica el ID de la tarea a marcar como completada
   Responde con JSON: {"accion": "completar", "id": numero}
4. ELIMINAR TAREA: Identifica el ID a eliminar
   Responde con JSON: {"accion": "eliminar", "id": numero}
5. RESUMEN/ESTADÍSTICAS: Analiza el resumen y da consejos de productividad
6. PREGUNTAS GENERALES: Responde con consejos de estudio, técnicas de gestión del tiempo, etc.

IMPORTANTE:
- Cuando ejecutes acciones (agregar/completar/eliminar), incluye el JSON en tu respuesta
- Sé amigable, motivador y conciso
- Usa emojis moderadamente
- Si el usuario habla en español, responde en español
- Siempre confirma las acciones realizadas
- Cuando listes tareas, usa formato claro con estado, materia y fecha
"""

class AgenteIATareas:
    def __init__(self):
        self.model = genai.GenerativeModel(
            model_name="gemini-1.5-flash",  # Modelo gratuito
            system_instruction=SYSTEM_PROMPT
        )
        self.historial = []
        self.chat = self.model.start_chat(history=[])

    def _construir_contexto(self):
        tareas = cargar_tareas()
        resumen = obtener_resumen()
        contexto = f"""
=== ESTADO ACTUAL DEL SISTEMA DE TAREAS ===
Fecha y hora actual: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}

RESUMEN:
- Total de tareas: {resumen['total']}
- Completadas: {resumen['completadas']}
- Pendientes: {resumen['pendientes']}
- Vencidas: {len(resumen['vencidas'])}
- Próximas 7 días: {len(resumen['proximas_7dias'])}

LISTA COMPLETA DE TAREAS:
{json.dumps(tareas, ensure_ascii=False, indent=2)}
===========================================
"""
        return contexto

    def _procesar_accion(self, respuesta_texto):
        import re
        # Buscar JSON en la respuesta
        json_match = re.search(r'\{[^{}]*"accion"[^{}]*\}', respuesta_texto, re.DOTALL)
        if not json_match:
            return None

        try:
            datos = json.loads(json_match.group())
            accion = datos.get("accion")

            if accion == "agregar":
                nueva = agregar_tarea(
                    titulo=datos.get("titulo", "Sin título"),
                    descripcion=datos.get("descripcion", ""),
                    fecha_entrega=datos.get("fecha_entrega", ""),
                    materia=datos.get("materia", "General"),
                    prioridad=datos.get("prioridad", "media")
                )
                return f"\n💾 [Sistema] Tarea #{nueva['id']} guardada: '{nueva['titulo']}'"

            elif accion == "completar":
                completada = completar_tarea(int(datos.get("id", 0)))
                if completada:
                    return f"\n✅ [Sistema] Tarea #{completada['id']} marcada como completada"
                return "\n⚠️ [Sistema] No se encontró la tarea"

            elif accion == "eliminar":
                eliminar_tarea(int(datos.get("id", 0)))
                return f"\n🗑️ [Sistema] Tarea #{datos['id']} eliminada"

        except Exception as e:
            return f"\n⚠️ Error al procesar acción: {e}"

        return None

    def chatear(self, mensaje_usuario):
        try:
            # Incluir contexto actualizado en cada mensaje
            contexto = self._construir_contexto()
            mensaje_completo = f"{contexto}\n\nMensaje del usuario: {mensaje_usuario}"

            respuesta = self.chat.send_message(mensaje_completo)
            texto_respuesta = respuesta.text

            # Procesar acciones automáticamente
            resultado_accion = self._procesar_accion(texto_respuesta)

            # Limpiar JSON de la respuesta para mostrar
            import re
            texto_limpio = re.sub(r'\{[^{}]*"accion"[^{}]*\}', '', texto_respuesta, flags=re.DOTALL).strip()

            if resultado_accion:
                texto_limpio += resultado_accion

            return texto_limpio

        except Exception as e:
            return f"❌ Error al comunicarse con la IA: {str(e)}"

# Crear instancia del agente
agente = AgenteIATareas()
print("✅ Agente IA ProfeBot listo para usar!")

✅ Agente IA ProfeBot listo para usar!


In [5]:
# ============================================================
# CELDA 5: Interfaz visual con ipywidgets
# ============================================================
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Estilos CSS
display(HTML("""
<style>
.chat-container {
    font-family: 'Segoe UI', sans-serif;
    max-width: 800px;
    margin: 0 auto;
}
.msg-user {
    background: #4F46E5;
    color: white;
    padding: 10px 14px;
    border-radius: 18px 18px 4px 18px;
    margin: 6px 0 6px 80px;
    font-size: 14px;
    line-height: 1.5;
}
.msg-bot {
    background: #F3F4F6;
    color: #111827;
    padding: 10px 14px;
    border-radius: 18px 18px 18px 4px;
    margin: 6px 80px 6px 0;
    font-size: 14px;
    line-height: 1.6;
    white-space: pre-wrap;
}
.msg-system {
    background: #ECFDF5;
    color: #065F46;
    padding: 6px 12px;
    border-radius: 8px;
    margin: 4px 40px;
    font-size: 12px;
    border-left: 3px solid #10B981;
}
.header-box {
    background: linear-gradient(135deg, #4F46E5, #7C3AED);
    color: white;
    padding: 16px 20px;
    border-radius: 12px;
    margin-bottom: 12px;
    text-align: center;
}
.stats-box {
    display: flex;
    gap: 10px;
    margin-bottom: 12px;
    flex-wrap: wrap;
}
.stat-card {
    flex: 1;
    min-width: 100px;
    background: white;
    border: 1px solid #E5E7EB;
    border-radius: 10px;
    padding: 10px;
    text-align: center;
}
.stat-num { font-size: 24px; font-weight: 700; color: #4F46E5; }
.stat-label { font-size: 11px; color: #6B7280; }
.suggestions {
    display: flex;
    gap: 6px;
    flex-wrap: wrap;
    margin: 8px 0;
}
.sugg-btn {
    background: #EEF2FF;
    color: #4338CA;
    border: 1px solid #C7D2FE;
    border-radius: 20px;
    padding: 4px 12px;
    font-size: 12px;
    cursor: pointer;
}
</style>
"""))

def generar_stats_html():
    r = obtener_resumen()
    pct = int((r['completadas'] / r['total'] * 100)) if r['total'] > 0 else 0
    return f"""
    <div class="stats-box">
        <div class="stat-card">
            <div class="stat-num">{r['total']}</div>
            <div class="stat-label">📋 Total</div>
        </div>
        <div class="stat-card">
            <div class="stat-num" style="color:#10B981">{r['completadas']}</div>
            <div class="stat-label">✅ Completadas</div>
        </div>
        <div class="stat-card">
            <div class="stat-num" style="color:#F59E0B">{r['pendientes']}</div>
            <div class="stat-label">⏳ Pendientes</div>
        </div>
        <div class="stat-card">
            <div class="stat-num" style="color:#EF4444">{len(r['vencidas'])}</div>
            <div class="stat-label">🚨 Vencidas</div>
        </div>
        <div class="stat-card">
            <div class="stat-num">{pct}%</div>
            <div class="stat-label">📊 Progreso</div>
        </div>
    </div>
    """

# Widgets
header_out = widgets.Output()
stats_out = widgets.Output()
chat_out = widgets.Output()

entrada = widgets.Text(
    placeholder='Escribe aquí tu mensaje... (ej: "Muéstrame mis tareas pendientes")',
    layout=widgets.Layout(width='100%', height='40px')
)
btn_enviar = widgets.Button(
    description='Enviar ↑',
    button_style='primary',
    layout=widgets.Layout(width='100px', height='40px')
)
btn_limpiar = widgets.Button(
    description='Limpiar',
    button_style='',
    layout=widgets.Layout(width='80px', height='40px')
)
btn_stats = widgets.Button(
    description='📊 Actualizar',
    button_style='info',
    layout=widgets.Layout(width='110px', height='34px')
)

# Sugerencias rápidas
sugerencias = [
    "Ver tareas pendientes",
    "¿Cuáles son mis tareas urgentes?",
    "Agregar nueva tarea",
    "Dame consejos de estudio",
    "Mostrar resumen del curso",
    "¿Qué debo hacer hoy?"
]

mensajes_chat = []

def actualizar_stats():
    with stats_out:
        clear_output(wait=True)
        display(HTML(generar_stats_html()))

def agregar_mensaje_chat(tipo, texto):
    mensajes_chat.append((tipo, texto))
    with chat_out:
        clear_output(wait=True)
        html = '<div class="chat-container">'
        for t, msg in mensajes_chat:
            css_class = "msg-user" if t == "user" else ("msg-system" if t == "system" else "msg-bot")
            prefijo = "Tú: " if t == "user" else ("" if t == "system" else "🎓 ProfeBot: ")
            msg_escaped = msg.replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
            html += f'<div class="{css_class}">{prefijo}{msg_escaped}</div>'
        html += '</div>'
        display(HTML(html))

def enviar_mensaje(b=None):
    msg = entrada.value.strip()
    if not msg:
        return
    entrada.value = ''
    agregar_mensaje_chat("user", msg)
    agregar_mensaje_chat("bot", "⏳ Pensando...")
    respuesta = agente.chatear(msg)
    mensajes_chat.pop()  # Quitar "Pensando..."
    agregar_mensaje_chat("bot", respuesta)
    actualizar_stats()

def limpiar_chat(b):
    mensajes_chat.clear()
    global agente
    agente = AgenteIATareas()  # Reiniciar historial
    with chat_out:
        clear_output()
    agregar_mensaje_chat("system", "Chat reiniciado ✨")

def enviar_sugerencia(texto):
    entrada.value = texto
    enviar_mensaje()

def on_enter(change):
    if hasattr(change, 'new') and '\n' not in str(change.new):
        pass

entrada.on_submit(enviar_mensaje)
btn_enviar.on_click(enviar_mensaje)
btn_limpiar.on_click(limpiar_chat)
btn_stats.on_click(lambda b: actualizar_stats())

# Renderizar header
with header_out:
    display(HTML("""
    <div class="header-box">
        <h2 style="margin:0;font-size:22px">🎓 ProfeBot — Agente IA de Tareas</h2>
        <p style="margin:4px 0 0;font-size:13px;opacity:0.85">Tu asistente inteligente para gestionar las tareas del curso</p>
    </div>
    """))

# Botones de sugerencias
sugg_html = '<div class="suggestions">' + ''.join(
    f'<span class="sugg-btn" onclick="">💬 {s}</span>' for s in sugerencias
) + '</div>'

btns_sugerencias = []
for s in sugerencias:
    btn = widgets.Button(
        description=f'💬 {s}',
        button_style='',
        layout=widgets.Layout(height='30px')
    )
    texto_capturado = s
    btn.on_click(lambda b, t=texto_capturado: enviar_sugerencia(t))
    btns_sugerencias.append(btn)

fila_sugerencias1 = widgets.HBox(btns_sugerencias[:3])
fila_sugerencias2 = widgets.HBox(btns_sugerencias[3:])

# Layout final
barra_entrada = widgets.HBox(
    [entrada, btn_enviar, btn_limpiar],
    layout=widgets.Layout(width='100%', gap='6px')
)

display(header_out)

stats_label = widgets.HBox([
    widgets.HTML('<b style="font-size:13px;color:#374151">📊 Estadísticas del curso:</b>'),
    btn_stats
], layout=widgets.Layout(justify_content='space-between', align_items='center'))
display(stats_label)
display(stats_out)
actualizar_stats()

display(widgets.HTML('<p style="font-size:12px;color:#6B7280;margin:4px 0">⚡ Sugerencias rápidas:</p>'))
display(fila_sugerencias1)
display(fila_sugerencias2)
display(widgets.HTML('<hr style="border:none;border-top:1px solid #E5E7EB;margin:10px 0">'))
display(widgets.HTML('<p style="font-size:13px;color:#374151;font-weight:500">💬 Chat con ProfeBot:</p>'))
display(chat_out)
display(barra_entrada)

# Mensaje de bienvenida
agregar_mensaje_chat("bot", "¡Hola! Soy ProfeBot 🎓, tu asistente para gestionar las tareas del curso.\n\nPuedo ayudarte a:\n• Ver y organizar tus tareas pendientes\n• Agregar nuevas tareas con fechas y prioridades\n• Marcar tareas como completadas\n• Darte consejos de estudio y gestión del tiempo\n\n¿Por dónde quieres empezar?")

Output()

Output()

HTML(value='<p style="font-size:12px;color:#6B7280;margin:4px 0">⚡ Sugerencias rápidas:</p>')

HTML(value='<hr style="border:none;border-top:1px solid #E5E7EB;margin:10px 0">')

HTML(value='<p style="font-size:13px;color:#374151;font-weight:500">💬 Chat con ProfeBot:</p>')

Output()

In [ ]:
# ============================================================
# CELDA 6 (OPCIONAL): Comandos directos sin chat
# ============================================================
# Puedes usar estas funciones directamente si prefieres

# Ver todas las tareas
def ver_tareas_tabla():
    tareas = cargar_tareas()
    print(f"{'ID':<4} {'TÍTULO':<30} {'MATERIA':<15} {'FECHA':<12} {'PRIORIDAD':<10} {'ESTADO':<12}")
    print("-" * 85)
    for t in tareas:
        estado_icon = "✅" if t['estado'] == 'completada' else "⏳"
        prioridad_icon = {"alta": "🔴", "media": "🟡", "baja": "🟢"}.get(t.get("prioridad","media"), "⚪")
        print(f"{t['id']:<4} {t['titulo'][:29]:<30} {t.get('materia','')[:14]:<15} {t.get('fecha_entrega',''):<12} {prioridad_icon+' '+t.get('prioridad',''):<10} {estado_icon+' '+t['estado']:<12}")

# Descomentar para usar:
# ver_tareas_tabla()

# Agregar tarea manualmente:
# agregar_tarea("Mi Tarea", "Descripción", "2025-04-30", "Materia", "alta")

# Marcar como completada (por ID):
# completar_tarea(1)

# Guardar tareas en Google Drive (opcional):
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('tareas_curso.json', '/content/drive/MyDrive/tareas_curso.json')
# print('✅ Tareas guardadas en Google Drive')

print("✅ Comandos disponibles. Descomenta las líneas que quieras usar.")

---
## 📖 Guía de uso rápida

### Comandos que puedes decirle al agente:

| Lo que dices | Lo que hace |
|---|---|
| `"Muéstrame todas mis tareas"` | Lista todas las tareas con estado |
| `"¿Qué tareas tengo esta semana?"` | Filtra tareas próximas |
| `"Agrega una tarea de Matemáticas para el viernes"` | Crea la tarea automáticamente |
| `"Marca la tarea 3 como completada"` | Actualiza el estado |
| `"¿Cuál debería hacer primero?"` | Priorización con IA |
| `"Dame un plan de estudio para esta semana"` | Genera plan personalizado |
| `"Eliminar tarea 5"` | Borra una tarea |

### 💾 Persistencia de datos
- Las tareas se guardan en `tareas_curso.json` en el entorno de Colab
- Para persistir entre sesiones, guarda el archivo en **Google Drive** (ver Celda 6)

### 🆓 Costo
- **Google Gemini Flash**: 15 RPM y 1 millón de tokens/día en capa gratuita
- Más que suficiente para uso académico diario
